# 7 Bilingual PySpark: Blending Python and Sql Code

Diagram comparing logic structure between pyspark and sql
![pysparkVsSql.png](media/pysparkVsSql.png)

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
import pyspark.sql.functions as F 
import pyspark.sql.types as T 
import logging

spark = SparkSession.builder.getOrCreate()
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [2]:
# create dataframe
elements = spark.read.csv("data/elements/Periodic_Table_Of_Elements.csv", header=True, inferSchema=True)
elements.show()

25/01/05 09:50:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+----------+------+----------+----------------+---------------+-----------------+------+-----+-----+-----------+-------+-----+--------+---------+--------------------+------------+-----------------+---------------+-------+------------+------------+----------------+-------------------+----+------------+--------------+---------------+
|AtomicNumber|   Element|Symbol|AtomicMass|NumberofNeutrons|NumberofProtons|NumberofElectrons|Period|Group|Phase|Radioactive|Natural|Metal|Nonmetal|Metalloid|                Type|AtomicRadius|Electronegativity|FirstIonization|Density|MeltingPoint|BoilingPoint|NumberOfIsotopes|         Discoverer|Year|SpecificHeat|NumberofShells|NumberofValence|
+------------+----------+------+----------+----------------+---------------+-----------------+------+-----+-----+-----------+-------+-----+--------+---------+--------------------+------------+-----------------+---------------+-------+------------+------------+----------------+-------------------+----+----

In [4]:
elements.where(F.col("phase") == "liq").groupby("period").count().show()

+------+-----+
|period|count|
+------+-----+
|     6|    1|
|     4|    1|
+------+-----+



In [13]:
# to use sql to work with pyspark dataframes, one must register the dataframe as sql enabled with a method like createOrReplaceTempView() in the spark.catalog
# if you don't, you get an error like this example
try:
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
except AnalysisException as e:
    logger.info(f"Expected error thrown: {e}", e)

--- Logging error ---
Traceback (most recent call last):
  File "/tmp/ipykernel_7225/2618172488.py", line 4, in <module>
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/pyspark/sql/session.py", line 1631, in sql
    return DataFrame(self._jsparkSession.sql(sqlQuery, litArgs), self)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/home/hubert/data-analysis-pyspark/.venv/lib/python3.11/site-packages/pyspark/errors/exceptions/captured.py", line 185, in deco
    raise converted from None
pyspark.errors.exceptions.captured.AnalysisException: [TABLE_OR_VI

In [25]:
# adding the element data frame to the spark catalog so it can be used via sql queries
"""
Here we are adding the elements dataframe to the spark catalog via `createOrReplaceTempView` which is one of a family of functions that include:
 - createTempView
 - createOrReplaceTempView
 - createGlobalTempView
 - createOrReplaceGlobalTempView

 The 'createOrReplace' prefix means that the function method will allow you to overwrite existing tempViews with the same name
 while the just `create` prefix means that the function method will throw an error if an overwrite is about to happen.
 
 A "global" temp view is one that is shared accross multiple spark sessions.

"""
elements.createOrReplaceTempView("elements")

try:
    spark.sql("""select period, count(*) from elements where phase='liq' group by period""").show()
except AnalysisException as e:
    logger.info(f"Expected error thrown: {e}", e)

+------+--------+
|period|count(1)|
+------+--------+
|     6|       1|
|     4|       1|
+------+--------+



In [28]:
"""
You can view and manage views in the spark catalog via `spark.catalog`
"""
# reset tables
elements.createOrReplaceTempView("elements")
# list views
tblList1 = spark.catalog.listTables()
assert len(tblList1) == 1
logger.info(f"table list: {tblList1}")

INFO:__main__:table list: [Table(name='elements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


In [29]:
# delete a view
spark.catalog.dropTempView("elements")
# verify that the view is deleted
tblListPostDrop = spark.catalog.listTables()
assert len(tblListPostDrop) == 0
logger.info(f"tbl list post drop: {tblListPostDrop}")

INFO:__main__:tbl list post drop: []
